### import


In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance,PayloadSchemaType,PointStruct,MatchAny,FieldCondition,Filter,Prefetch, FusionQuery

import pandas as pd
import numpy as np
import openai
import json
import tiktoken

c:\Users\Loq\Documents\CRAP\end to end aibootcamp\code\handsON\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Retrive all item ids from amazon items qdrant collections

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")



In [4]:
dummy_vector = np.zeros(1536).tolist()


In [5]:
payload=qdrant_client.query_points(
        collection_name="amazon-items-collection-02-openai-small",
        
        query=dummy_vector,
        using="text-embedding-3-small",
        limit=1000,
        with_payload=["parent_asin"],
        with_vectors=False,
)


In [6]:
payload.points

[ScoredPoint(id=63, version=1, score=0.0, payload={'parent_asin': 'B09V2TBVHV'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=28, version=1, score=0.0, payload={'parent_asin': 'B0BJFPFR2Z'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=44, version=1, score=0.0, payload={'parent_asin': 'B0B4KFSLVW'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4, version=1, score=0.0, payload={'parent_asin': 'B0BS1SHT91'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=29, version=1, score=0.0, payload={'parent_asin': 'B09R84DXR1'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=13, version=1, score=0.0, payload={'parent_asin': 'B0B5B8JP2T'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=53, version=1, score=0.0, payload={'parent_asin': 'B004GPEB50'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, version=1, score=0.0, payload={'parent_asin': 'B09Q98326Z'}, vector=None, shard

In [7]:
len(payload.points)

1000

In [8]:
parent_asin_list = [point.payload["parent_asin"] for point in payload.points]

In [9]:
parent_asin_list

['B09V2TBVHV',
 'B0BJFPFR2Z',
 'B0B4KFSLVW',
 'B0BS1SHT91',
 'B09R84DXR1',
 'B0B5B8JP2T',
 'B004GPEB50',
 'B09Q98326Z',
 'B09TGDWJRP',
 'B0BPN1K83D',
 'B0B14G1W9T',
 'B09R1C2F83',
 'B00A2K7LG4',
 'B09RQGFDDW',
 'B0BL4VY2NB',
 'B0BTW59YJJ',
 'B0BS48V39J',
 'B0B1QVBDSK',
 'B09SP9JXV9',
 'B09YTJPYM4',
 'B0BJ35953K',
 'B0B91C6BSR',
 'B09NBJJGGX',
 'B09VGGZKFN',
 'B0BJVZQB6S',
 'B0BJ1CVBQY',
 'B0B2HTZ54Z',
 'B0BN8LTKRD',
 'B0BSNRV2Z3',
 'B09PP8QPTZ',
 'B0BG3F848Q',
 'B09SXRT2LR',
 'B09TFVBKNH',
 'B09S9WTGX7',
 'B0B225YLCD',
 'B0B4P55X7P',
 'B0B2X5W3YH',
 'B09TN23LF5',
 'B0B6GGYWMS',
 'B0BWFRVVT2',
 'B0BHFKN43K',
 'B09S5JJDFQ',
 'B0B8DWKX13',
 'B09ZKJJ62H',
 'B0BFP93P9T',
 'B09Q12DWG2',
 'B0BD4XPYYM',
 'B0B94TZTZ6',
 'B09X257ZY5',
 'B0B6J6632J',
 'B0B18X4SKS',
 'B0B4L1SZRY',
 'B0B1DW3VP6',
 'B09Q962VN1',
 'B0BLJM97W1',
 'B0BFJXBYV1',
 'B0BPMN3P74',
 'B0BD5X3S6J',
 'B0BDGB3YRJ',
 'B0B3QX9JMM',
 'B0B3JS5743',
 'B077P4L1M4',
 'B0B8CKYDTS',
 'B0BS1VVXT1',
 'B09VYP9LRG',
 'B09ZN6YVV8',
 'B0BG9HWG

#### load amazon reviews dataset


In [10]:
df_reviews=pd.read_json("../../data/CDs_and_Vinyl_2022_2023_with_category_ratings_10_sample_1000.jsonl",lines=True)

In [11]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,"If this is rock 'n' roll retirement, I'll take...","Yet again, David Crosby keeps the winning stre...",[],B0BH3R9D4X,B0BH3R9D4X,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2023-01-08 22:56:15.968,8,True
1,4,"CCR's legendary 1970 Royal Albert Hall gig, fi...",CCR's legendary 1970 Royal Albert Hall concert...,[],B0B18Z8GL1,B0B18Z8GL1,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-09-16 18:05:34.559,21,True
2,3,Songs for Beginners / Wild Tales 50 + years do...,So many 5-star reviews. I just don't get it. ...,[],B09VCS9Q5W,B09VCS9Q5W,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-06-03 20:56:32.958,4,True
3,4,A master lesson from a pair of legends,Mavis and Levon. Staples and Helm. If you're...,[],B09VLCNB1F,B09VLCNB1F,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-05-22 15:32:49.839,9,True
4,3,"Disappointing, at best","Musically and vocally, Van's still the man. B...",[],B09V4R82C4,B09V4R82C4,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-05-22 15:03:02.428,1,True


In [12]:
len(df_reviews)

8721

In [13]:
df_reviews_sample=df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]

In [14]:
len(df_reviews_sample)

8721

#### define funtions to preprocess reviews data

In [15]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"


In [16]:
encoding=tiktoken.encoding_for_model("text-embedding-3-small")

In [17]:
encoding.encode("Can I get some earphones")

[6854, 358, 636, 1063, 2487, 17144]

In [18]:
def token_count(row,model="text-embedding-3-small"):
    encoding=tiktoken.encoding_for_model(model)
    return len(encoding.encode(row["preprocessed_data"]))

In [19]:
df_reviews_sample["preprocessed_data"]=df_reviews_sample.apply(preprocess_reviews_data,axis=1)

In [20]:
df_reviews_sample["preprocessed_data_token_count"]=df_reviews_sample.apply(token_count,axis=1)

In [21]:
df_reviews_sample.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,preprocessed_data_token_count
0,4,"If this is rock 'n' roll retirement, I'll take...","Yet again, David Crosby keeps the winning stre...",[],B0BH3R9D4X,B0BH3R9D4X,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2023-01-08 22:56:15.968,8,True,"If this is rock 'n' roll retirement, I'll take...",210
1,4,"CCR's legendary 1970 Royal Albert Hall gig, fi...",CCR's legendary 1970 Royal Albert Hall concert...,[],B0B18Z8GL1,B0B18Z8GL1,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-09-16 18:05:34.559,21,True,"CCR's legendary 1970 Royal Albert Hall gig, fi...",321
2,3,Songs for Beginners / Wild Tales 50 + years do...,So many 5-star reviews. I just don't get it. ...,[],B09VCS9Q5W,B09VCS9Q5W,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-06-03 20:56:32.958,4,True,Songs for Beginners / Wild Tales 50 + years do...,355
3,4,A master lesson from a pair of legends,Mavis and Levon. Staples and Helm. If you're...,[],B09VLCNB1F,B09VLCNB1F,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-05-22 15:32:49.839,9,True,A master lesson from a pair of legends Mavis a...,117
4,3,"Disappointing, at best","Musically and vocally, Van's still the man. B...",[],B09V4R82C4,B09V4R82C4,AE7BV6IMNPZ3F266H7PXMH3BZQNQ,2022-05-22 15:03:02.428,1,True,"Disappointing, at best Musically and vocally, ...",90


In [22]:
len(df_reviews_sample)

8721

In [23]:
df_reviews_sample=df_reviews_sample[df_reviews_sample["preprocessed_data_token_count"]<8192]

In [24]:
len(df_reviews_sample)

8721

In [25]:
total_tokens=df_reviews_sample["preprocessed_data_token_count"].sum()

In [26]:
total_tokens

np.int64(562054)

#### create a new Qdrant Collection for reviews


In [27]:
qdrant_client.create_collection(
    collection_name="amazon-items-collection-02-openai-small-reviews",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `amazon-items-collection-02-openai-small-reviews` already exists!"},"time":0.000401292}'

In [28]:
qdrant_client.create_payload_index(
    collection_name="amazon-items-collection-02-openai-small-reviews",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

#### embedding functions

In [29]:
def get_embedding(text, model="text-embedding-3-small"):
    response=openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding

In [30]:
def get_embedding_batch(text_list, model="text-embedding-3-small",batch_size=100):
    if len(text_list)<=batch_size:
        response=openai.embeddings.create(
            input=text_list,
            model=model,
        )
        return [item.embedding for item in response.data]

    all_embeddings=[]
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch=text_list[i:i+batch_size]
        response=openai.embeddings.create(
            input=batch,
            model=model,
        )
        all_embeddings.extend([item.embedding for item in response.data])
        print(f"Processed batch {counter} of {len(text_list)//batch_size + 1}")
        counter += 1
    return all_embeddings

#### embed the text and add additional fields to the payload of each vector for reviews

In [31]:
data_to_embed_reviews=df_reviews_sample[["preprocessed_data","parent_asin"]].to_dict(orient="records")

In [32]:
data_to_embed_reviews

[{'preprocessed_data': 'If this is rock \'n\' roll retirement, I\'ll take it! Yet again, David Crosby keeps the winning streak alive.  Live at the Capitol Theatre.  Incredibly, and against all kinds of odds, Croz has outlasted and outpaced (with the possible exception of Chris Hillman) all of his former Byrds and CSN (&Y) bandmates since crossing over into the new millennium.  And don\'t forget "& the Lighthouse Band."  This is a group effort, and it\'s a solid one.  Covering material from across Crosby\'s storied career, with a definite accent on the more recent works, Live at the Capitol is aural technicolor dream, a kind of band biography in song.  Don\'t know if you had to be there, or not -- I wasn\'t -- but you\'ll feel like you\'re right there in the audience whether listening to the CD or watching and listening to the DVD.  Very impressive live album.  Four shining stars and definitely recommended.',
  'parent_asin': 'B0BH3R9D4X'},
 {'preprocessed_data': "CCR's legendary 1970 R

In [33]:
text_to_embed_reviews=[item["preprocessed_data"] for item in data_to_embed_reviews]

In [34]:
text_to_embed_reviews

['If this is rock \'n\' roll retirement, I\'ll take it! Yet again, David Crosby keeps the winning streak alive.  Live at the Capitol Theatre.  Incredibly, and against all kinds of odds, Croz has outlasted and outpaced (with the possible exception of Chris Hillman) all of his former Byrds and CSN (&Y) bandmates since crossing over into the new millennium.  And don\'t forget "& the Lighthouse Band."  This is a group effort, and it\'s a solid one.  Covering material from across Crosby\'s storied career, with a definite accent on the more recent works, Live at the Capitol is aural technicolor dream, a kind of band biography in song.  Don\'t know if you had to be there, or not -- I wasn\'t -- but you\'ll feel like you\'re right there in the audience whether listening to the CD or watching and listening to the DVD.  Very impressive live album.  Four shining stars and definitely recommended.',
 "CCR's legendary 1970 Royal Albert Hall gig, finally CCR's legendary 1970 Royal Albert Hall concert

In [35]:
embeddings_reviews=get_embedding_batch(text_to_embed_reviews,batch_size=500)

Processed batch 1 of 18
Processed batch 2 of 18
Processed batch 3 of 18
Processed batch 4 of 18
Processed batch 5 of 18
Processed batch 6 of 18
Processed batch 7 of 18
Processed batch 8 of 18
Processed batch 9 of 18
Processed batch 10 of 18
Processed batch 11 of 18
Processed batch 12 of 18
Processed batch 13 of 18
Processed batch 14 of 18
Processed batch 15 of 18
Processed batch 16 of 18
Processed batch 17 of 18
Processed batch 18 of 18


In [36]:
len(embeddings_reviews)

8721

In [37]:
pointstructs=[]
i=1
for embedding, data in zip(embeddings_reviews, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload={
                "text": data["preprocessed_data"],
                "parent_asin": data["parent_asin"]
            }
        )
    )
    i += 1

In [38]:
batch_size_qdrant=100
counter=1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch=pointstructs[i:i+batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="amazon-items-collection-02-openai-small-reviews",
        wait=True,
        points=batch
    )
    print(f"Processed batch {counter} of {len(pointstructs)//batch_size_qdrant + 1}")
    counter += 1

Processed batch 1 of 88
Processed batch 2 of 88
Processed batch 3 of 88
Processed batch 4 of 88
Processed batch 5 of 88
Processed batch 6 of 88
Processed batch 7 of 88
Processed batch 8 of 88
Processed batch 9 of 88
Processed batch 10 of 88
Processed batch 11 of 88
Processed batch 12 of 88
Processed batch 13 of 88
Processed batch 14 of 88
Processed batch 15 of 88
Processed batch 16 of 88
Processed batch 17 of 88
Processed batch 18 of 88
Processed batch 19 of 88
Processed batch 20 of 88
Processed batch 21 of 88
Processed batch 22 of 88
Processed batch 23 of 88
Processed batch 24 of 88
Processed batch 25 of 88
Processed batch 26 of 88
Processed batch 27 of 88
Processed batch 28 of 88
Processed batch 29 of 88
Processed batch 30 of 88
Processed batch 31 of 88
Processed batch 32 of 88
Processed batch 33 of 88
Processed batch 34 of 88
Processed batch 35 of 88
Processed batch 36 of 88
Processed batch 37 of 88
Processed batch 38 of 88
Processed batch 39 of 88
Processed batch 40 of 88
Processed

#### function to run search against reviews on a prefiltered set of product IDs

In [39]:
def retrieve_prefiltered_reviews_data(query, parent_asins,k=5):
    query_embedding=get_embedding(query)
    results=qdrant_client.query_points(
        collection_name="amazon-items-collection-02-openai-small-reviews",
        prefetch=[
            Prefetch(
                query=query_embedding,
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )
    return results

#### the test say a review with bad quality in a set of parent-asins

In [42]:
reviews=retrieve_prefiltered_reviews_data("bad quality", ["B09F1FJCH6"],k=5)

In [44]:
reviews.points

[ScoredPoint(id=2680, version=31, score=0.5, payload={'text': 'So so quality Quality is not what I want', 'parent_asin': 'B09F1FJCH6'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4672, version=51, score=0.33333334, payload={'text': 'Pressing Not Very Good This is a great album but the pressing isn’t so great. A lot of particles in the vinyl. If you’re picky about vinyl quality you will probably be disappointed.', 'parent_asin': 'B09F1FJCH6'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4103, version=46, score=0.25, payload={'text': 'Inexpensive but not 180g Sounds good, but not an audiophile quality pressing by any means.', 'parent_asin': 'B09F1FJCH6'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=8277, version=87, score=0.2, payload={'text': 'Sony Music, get your QC together Highway 61 Revisited sleeve, Highway 61 Revisited label, but the actual record was Bringing It All Back Home. Nonexistent QC. Sony more concerned with pr

In [45]:
for point in reviews.points:
    print(point.payload["parent_asin"])
    print(point.payload["text"])
    print("="*100)


B09F1FJCH6
So so quality Quality is not what I want
B09F1FJCH6
Pressing Not Very Good This is a great album but the pressing isn’t so great. A lot of particles in the vinyl. If you’re picky about vinyl quality you will probably be disappointed.
B09F1FJCH6
Inexpensive but not 180g Sounds good, but not an audiophile quality pressing by any means.
B09F1FJCH6
Sony Music, get your QC together Highway 61 Revisited sleeve, Highway 61 Revisited label, but the actual record was Bringing It All Back Home. Nonexistent QC. Sony more concerned with pressing another 100K Adele albums?<br /><br />Follow-up: Replacement record received. This time, it was the correct record.
B09F1FJCH6
Warped disc I love Bob Dylan , but do not like warped records; it went back
